In [1]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv('../data/raw/telecom_data.csv')

# Handle missing values
df['TCP DL Retrans. Vol (Bytes)'] = df['TCP DL Retrans. Vol (Bytes)'].fillna(df['TCP DL Retrans. Vol (Bytes)'].mean())
df['TCP UL Retrans. Vol (Bytes)'] = df['TCP UL Retrans. Vol (Bytes)'].fillna(df['TCP UL Retrans. Vol (Bytes)'].mean())
df['Avg RTT DL (ms)'] = df['Avg RTT DL (ms)'].fillna(df['Avg RTT DL (ms)'].mean())
df['Avg RTT UL (ms)'] = df['Avg RTT UL (ms)'].fillna(df['Avg RTT UL (ms)'].mean())
df['Handset Type'] = df['Handset Type'].fillna('Unknown')
df['Avg Bearer TP DL (kbps)'] = df['Avg Bearer TP DL (kbps)'].fillna(df['Avg Bearer TP DL (kbps)'].mean())
df['Avg Bearer TP UL (kbps)'] = df['Avg Bearer TP UL (kbps)'].fillna(df['Avg Bearer TP UL (kbps)'].mean())

# Save the preprocessed dataset
df.to_csv('../data/processed/xdr_data_preprocessed.csv', index=False)
print("Preprocessed dataset saved to 'data/processed/xdr_data_preprocessed.csv'")

Preprocessed dataset saved to 'data/processed/xdr_data_preprocessed.csv'


In [2]:
# Load the preprocessed dataset
df = pd.read_csv('../data/processed/xdr_data_preprocessed.csv')

# Aggregate metrics per customer
experience_metrics = df.groupby('MSISDN/Number').agg({
    'TCP DL Retrans. Vol (Bytes)': 'mean',  # Average TCP retransmission (DL)
    'TCP UL Retrans. Vol (Bytes)': 'mean',  # Average TCP retransmission (UL)
    'Avg RTT DL (ms)': 'mean',              # Average RTT (DL)
    'Avg RTT UL (ms)': 'mean',              # Average RTT (UL)
    'Handset Type': 'first',                # Handset type
    'Avg Bearer TP DL (kbps)': 'mean',      # Average throughput (DL)
    'Avg Bearer TP UL (kbps)': 'mean'       # Average throughput (UL)
}).rename(columns={
    'TCP DL Retrans. Vol (Bytes)': 'Avg TCP Retransmission (DL)',
    'TCP UL Retrans. Vol (Bytes)': 'Avg TCP Retransmission (UL)',
    'Avg Bearer TP DL (kbps)': 'Avg Throughput DL (kbps)',
    'Avg Bearer TP UL (kbps)': 'Avg Throughput UL (kbps)'
})

# Compute total average throughput (DL + UL)
experience_metrics['Avg Total Throughput (kbps)'] = (
    experience_metrics['Avg Throughput DL (kbps)'] + experience_metrics['Avg Throughput UL (kbps)']
)

# Display aggregated data
print("Aggregated Experience Metrics:")
display(experience_metrics.head())

Aggregated Experience Metrics:


,Avg TCP Retransmission (DL),Avg TCP Retransmission (UL),Avg RTT DL (ms),Avg RTT UL (ms),Handset Type,Avg Throughput DL (kbps),Avg Throughput UL (kbps),Avg Total Throughput (kbps)
MSISDN/Number,,,,,,,,
3.360100e+10,2.080991e+07,759658.664811,46.000000,0.000000,Huawei P20 Lite Huawei Nova 3E,37.0,39.0,76.0
3.360100e+10,2.080991e+07,759658.664811,30.000000,1.000000,Apple iPhone 7 (A1778),48.0,51.0,99.0
3.360100e+10,2.080991e+07,759658.664811,109.795706,17.662883,undefined,48.0,49.0,97.0
3.360101e+10,1.066000e+03,759658.664811,69.000000,15.000000,Apple iPhone 5S (A1457),204.0,44.0,248.0
3.360101e+10,1.507977e+07,390430.332406,57.000000,2.500000,Apple iPhone Se (A1723),20197.5,8224.5,28422.0


In [3]:
# Save aggregated experience metrics
experience_metrics.to_csv('../data/processed/experience_metrics.csv', index=True)

print("Aggregated experience metrics saved to 'data/processed/experience_metrics.csv'")

Aggregated experience metrics saved to 'data/processed/experience_metrics.csv'


In [4]:
# Group by handset type and compute metrics
handset_analysis = df.groupby('Handset Type').agg({
    'Avg Bearer TP DL (kbps)': 'mean',  # Average throughput (DL)
    'Avg Bearer TP UL (kbps)': 'mean',  # Average throughput (UL)
    'TCP DL Retrans. Vol (Bytes)': 'mean',  # Average TCP retransmission (DL)
    'TCP UL Retrans. Vol (Bytes)': 'mean'   # Average TCP retransmission (UL)
}).rename(columns={
    'Avg Bearer TP DL (kbps)': 'Avg Throughput DL (kbps)',
    'Avg Bearer TP UL (kbps)': 'Avg Throughput UL (kbps)',
    'TCP DL Retrans. Vol (Bytes)': 'Avg TCP Retransmission (DL)',
    'TCP UL Retrans. Vol (Bytes)': 'Avg TCP Retransmission (UL)'
})

# Compute total average throughput (DL + UL)
handset_analysis['Avg Total Throughput (kbps)'] = (
    handset_analysis['Avg Throughput DL (kbps)'] + handset_analysis['Avg Throughput UL (kbps)']
)

# Save handset-specific analysis
handset_analysis.to_csv('../data/processed/handset_analysis.csv', index=True)

print("Handset-specific analysis saved to 'data/processed/handset_analysis.csv'")

Handset-specific analysis saved to 'data/processed/handset_analysis.csv'


In [5]:
from sklearn.preprocessing import MinMaxScaler

# Select relevant metrics for normalization
experience_data = experience_metrics[['Avg TCP Retransmission (DL)', 'Avg RTT DL (ms)', 'Avg Total Throughput (kbps)']]

# Normalize data
scaler = MinMaxScaler()
normalized_experience = scaler.fit_transform(experience_data)

# Convert normalized data to DataFrame
normalized_experience_df = pd.DataFrame(normalized_experience, columns=experience_data.columns, index=experience_metrics.index)

# Save normalized experience data
normalized_experience_df.to_csv('../data/processed/normalized_experience_data.csv', index=True)

print("Normalized experience data saved to 'data/processed/normalized_experience_data.csv'")

Normalized experience data saved to 'data/processed/normalized_experience_data.csv'


In [6]:
from sklearn.cluster import KMeans

# Run K-Means clustering (k=3)
kmeans = KMeans(n_clusters=3, random_state=42)
experience_metrics['Experience Cluster'] = kmeans.fit_predict(normalized_experience_df)

# Save experience metrics with cluster labels
experience_metrics.to_csv('../data/processed/experience_metrics_with_clusters.csv', index=True)

print("Experience metrics with cluster labels saved to 'data/processed/experience_metrics_with_clusters.csv'")

Experience metrics with cluster labels saved to 'data/processed/experience_metrics_with_clusters.csv'
